# 15.9 怎麼鼓勵較均衡的使用？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：四位師傅都在，這張訂單只請兩位動手**

例中只派給A與C；未選中師傅的權重仍佔儲存空間。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/moe.svg")))

**先想一想：**負載最平均就一定任務最準確嗎？

負載平衡項鼓勵不要只用同一位。但它和任務loss可能衝突，過強會為平均而分錯單。

**把直覺寫成數學：**aux=E·Σ_e fraction_selected_e·mean_probability_e；選擇頻率stop-gradient。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.modern import MoEFFN

layer = MoEFFN(4, experts=3, top_k=2)
out, aux, chosen = layer(torch.randn(2, 4, 4))
task = out.square().mean()
total = task + 0.01 * aux
total.backward()
print(task.item(), aux.item(), chosen.tolist())

**如何讀結果：**系數0.01是示範配方，需用validation選擇；load、品質與時間一起記錄。

**只改一件事：**只把aux系數提高100倍。
